In [1]:
import numpy as np
from functools import partial
import soundfile as sf
from scipy.signal import spectrogram
from decord import VideoReader, gpu
import fastplotlib as fpl
from ipywidgets import VBox, HBox
import time
import cv2
from video_reader import LazyVideo
import os
from pathlib import Path

Unable to find extension: VK_EXT_physical_device_drm
No windowing system present. Using surfaceless platform
No config found!
No config found!
Max vertex attribute stride unknown. Assuming it is 2048
Max vertex attribute stride unknown. Assuming it is 2048
Max vertex attribute stride unknown. Assuming it is 2048


Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),NVIDIA A100-SXM4-40GB,DiscreteGPU,Vulkan,550.90.07
❌,NVIDIA A100-SXM4-40GB/PCIe/SSE2,Unknown,OpenGL,3.3.0 NVIDIA 550.90.07


Max vertex attribute stride unknown. Assuming it is 2048
Max vertex attribute stride unknown. Assuming it is 2048


In [2]:
# ==== Choose ====
exp_num = 237
file_num = 15

channel_numbers = [2,0,4,5]

base_path_audio = "/mnt/home/neurostatslab/ceph/saneslab_data/big_setup/experiment_237/concatenated_data_cam_mic_sync"
base_path_video = "/mnt/home/neurostatslab/ceph/saneslab_data/big_setup/experiment_237/concatenated_data_cam_mic_sync"

bouts = np.load(Path("/mnt/home/kkolar/data/gerbil_sandbox/237/bouts").joinpath(f"{file_num}.npy"))
extracted_specs = np.load(Path("/mnt/home/kkolar/data/gerbil_sandbox/237/specs").joinpath(f"{file_num}.npy"))

spec_height = 300
spec_width = 1500
vid_height = 600

In [13]:
# Functions
def make_specgram(audio, fps=125000):
    f, t, spec = spectrogram(audio, fs=fps, nfft=n_fft, nperseg=n_samples_bin, noverlap=n_samples_overlap,
                             return_onesided=True)
    # Remove the 0-frequency bin and flip the frequency axis so that high frequencies are at the top.
    f = f[1:][::-1]
    spec = np.flip(spec[1:], axis=0)
    spec = np.log(np.abs(spec) + 1e-12).astype(np.float32)
    return t, f, spec

def compute_bins_for_window(fs, nperseg, noverlap, window_sec):
    """Computes the number of time bins in a given window duration."""
    delta_t = (nperseg - noverlap) / fs  # Time per bin
    bins_for_window = int(window_sec / delta_t)  # Total bins for desired duration
    return bins_for_window


# === spectrogram variables ===
fps_audio = 125000
fps_video = 29.9976  # 30
n_fft = 512
n_samples_bin = 512
n_samples_overlap = 256
window_width_sec = 0.5  # choose

def get_spect_window(ev):
    t_frame_video = ev['t']
    #t_sec = t_frame_video / 30  # how can I take out the 30?
    t_sec = t_frame_video / fps_video

    window_duration = window_width_sec
    dt = t_spec[1] - t_spec[0]
    window_size = int(window_duration / dt)

    center_idx = np.searchsorted(t_spec, t_sec)
    half_window = window_size // 2

    start_idx = max(0, center_idx - half_window)
    end_idx = min(len(t_spec), center_idx + half_window)

    # print(f"Frame: {t_frame_video}, Time (sec): {t_sec}, Index: {center_idx}, Window: ({start_idx}, {end_idx})")

    spec_slices: dict[str, np.ndarray] = dict()

    for loc in location_order:
        spec_data = specs[loc]

        spectrogram_slice = spec_data[:, start_idx:end_idx]

        # Pad if necessary to maintain fixed size
        if spectrogram_slice.shape[1] < window_size:
            pad_width = window_size - spectrogram_slice.shape[1]
            spectrogram_slice = np.pad(
                spectrogram_slice, ((0, 0), (0, pad_width)), mode="constant"
            )

        spec_slices[loc] = spectrogram_slice

    for loc in location_order:
        spec_fig[loc].graphics[0].data = spec_slices[loc]

    start_time = t_sec - (window_duration / 2)
    end_time = t_sec + (window_duration / 2)

    start_positions = bouts[:, 0][(bouts[:, 0] > start_time) & (bouts[:, 0] < end_time)]
    end_positions = bouts[:, 1][(bouts[:, 1] > start_time) & (bouts[:, 1] < end_time)]
    
    for subplot in spec_fig:
        for g in subplot.graphics:
            if isinstance(g, fpl.LineGraphic):
                subplot.delete_graphic(g)
    
    # 5 seconds is 2441 columns in the spectrogram
    # 488.2 cols/sec
    # position to draw line:
    for n, position in enumerate(start_positions):
        x_line = (position - start_time) * (spectrogram_slice.shape[1] / window_duration)
        for subplot in spec_fig:
            subplot.add_line(np.array([[x_line, 0], [x_line, 256]], dtype=np.float32), colors="m", name=f"start-{n}")
    
    for n, position in enumerate(end_positions):
        x_line = (position - start_time) * 488.22441
        for subplot in spec_fig:
            subplot.add_line(np.array([[x_line, 0], [x_line, 256]], dtype=np.float32), colors="cyan", name=f"end-{n}")
            
# === Video file naming rule based on channel number ===
video_prefix_map = {
    0: "video_center_",
    1: "video_center_",
    2: "video_gily_center_", # change back
    3: "video_gily_center_", # change back
    4: "video_nest_top_",     # "video_nest_top_", video_nest_side_
    5: "video_burrow_side_" #"video_burrow_top_" video_burrow_side_
}

# maps int -> location
name_mapping = {
    0: "center-1",
    1: "center-1", #1
    2: "center-2",
    3: "center-2",
    4: "nest",
    5: "burrow"
}

#location_order = ["center-2","center-1", "burrow", "nest"] # ["center-2", "center-1", "burrow", "nest"]
location_order =  ["center-2", "center-1", "burrow", "nest"]




# for all synchrinozed files
num_bins_window = compute_bins_for_window(fps_audio, n_samples_bin, n_samples_overlap, window_width_sec)

# === Collect paths ===
# maps location str -> path str
video_paths: dict[str, str] = dict()
audio_paths: dict[str, str] = dict()

# file_num_str_tmp = f"{file_num:0d}"
# file_num_str = f"{file_num:03d}"

file_num_str = file_num_str_tmp = str(file_num)

print(file_num_str_tmp)
for ch in channel_numbers:
    video_prefix = video_prefix_map[ch]
    if video_prefix is None:
        print(f"Warning: Invalid channel number {ch}, skipping.")
        continue

    video_path = os.path.join(base_path_video, f"{video_prefix}{file_num_str_tmp}.mp4") #####
    print(video_path)
    audio_path = os.path.join(base_path_audio, f"channel_{ch}_{file_num_str}.wav")



    video_paths[name_mapping[ch]] = video_path
    audio_paths[name_mapping[ch]] = audio_path


## Load video and audio ##
# maps location name -> LazyVideo
movies: dict[str, LazyVideo] = dict()

# maps location name -> full spectrogram
specs: dict[str, np.ndarray] = dict()

# gpu_context = gpu(0)

# --- Video loading ---
for location, path in video_paths.items():
    movies[location] = LazyVideo(path)

## --- trimming videos ---


## -----------------------


for location, path in audio_paths.items():
    audio_data, fps_audio = sf.read(path, dtype='float32')
    print(f"loaded: {path}")

    # Unpack the outputs of make_specgram
    t_spec, f_spec, spec_data = make_specgram(audio_data, fps_audio)

    specs[location] = spec_data




# Create the widget
spec_fig = fpl.Figure(
    size=(spec_width, spec_height),
    shape=(1, 4), #shape=(1, 4),
    names=[location_order],
    # controller_ids="sync"
)

for loc in location_order:
    spec_fig[loc].add_image(
        data=specs[loc][:, :num_bins_window],
        cmap="viridis",
        name="spectrogram",
    )
    spec_fig[loc].toolbar = False
    spec_fig[loc].camera.local.scale_y *= -1
    spec_fig[loc].axes.y.tick_format = lambda v, min_v, max_v: f"{v}"

n_ticks = 6

tmp_movies = [movies[loc] for loc in location_order]

#find the minimum number of frames across each movie
min_nf = []
for movie in tmp_movies:
    shape = movie.shape[0]
    min_nf.append(shape)
min_shape = min(min_nf)

# set each movies number of frames to the minimum
for movie in tmp_movies:
    movie.shape = (min_shape, movie.shape[1], movie.shape[2])

video_widget = fpl.ImageWidget(
    #[movies[loc] for loc in location_order],
    tmp_movies,
    histogram_widget=False,
    rgb=False,
    figure_kwargs={"size": (spec_width, vid_height), "shape": (1, 4), "controller_ids": None},
    names=location_order
)

for i, iw in enumerate([video_widget]):
    for subplot in iw.figure:
        subplot.toolbar = False
        subplot.axes.visible = False


    iw.add_event_handler(get_spect_window, "current_index")
    iw.show()


for iw in [video_widget]:
    for subplot in iw.figure:
        subplot.camera.zoom = 1.25

def update_iw_index(index):
    video_widget.current_index = index

#video_widget.figure.guis["bottom"].size = 0
video_widget.figure.guis["bottom"]
video_widget.add_event_handler(update_iw_index, "current_index")

VBox([spec_fig.show(maintain_aspect=False), video_widget.show()])

15
/mnt/home/neurostatslab/ceph/saneslab_data/big_setup/experiment_237/concatenated_data_cam_mic_sync/video_gily_center_15.mp4
/mnt/home/neurostatslab/ceph/saneslab_data/big_setup/experiment_237/concatenated_data_cam_mic_sync/video_center_15.mp4
/mnt/home/neurostatslab/ceph/saneslab_data/big_setup/experiment_237/concatenated_data_cam_mic_sync/video_nest_top_15.mp4
/mnt/home/neurostatslab/ceph/saneslab_data/big_setup/experiment_237/concatenated_data_cam_mic_sync/video_burrow_side_15.mp4
loaded: /mnt/home/neurostatslab/ceph/saneslab_data/big_setup/experiment_237/concatenated_data_cam_mic_sync/channel_2_15.wav
loaded: /mnt/home/neurostatslab/ceph/saneslab_data/big_setup/experiment_237/concatenated_data_cam_mic_sync/channel_0_15.wav
loaded: /mnt/home/neurostatslab/ceph/saneslab_data/big_setup/experiment_237/concatenated_data_cam_mic_sync/channel_4_15.wav
loaded: /mnt/home/neurostatslab/ceph/saneslab_data/big_setup/experiment_237/concatenated_data_cam_mic_sync/channel_5_15.wav


RFBOutputContext()

RFBOutputContext()